# Seed demo data — `wearables_zerobus` (bronze)

Inserts **Apple HealthKit–shaped** synthetic rows into Unity Catalog bronze (catalog/schema from widgets).

**Multi-user:** set **multi_user_mode** to `primary_plus_demo` (default) to load the signed-in user plus six stable demo personas—best for **Lakeview** charts split by `user_id`. Use `single` for one subject only, or `demo_cohort_only` for demo personas alone.

**Typical flow:** run this notebook → **run the bundle job `[dev] dbxWearables — Refresh wearable medallion DLT`** (or trigger an update on that pipeline in the UI) → open the **Lakeview** dashboard. The dashboard reads **gold** tables; if you only seed bronze, charts stay stale until the pipeline runs.

**History depth:** **history_days** (default 60) controls how many calendar days of activity summaries, sleep nights, and sample timestamps are synthesized—enough for trend charts on the dashboard.

Rows set `headers.x-ingest-channel = notebook_simulator` (and `x-device-id = demo-notebook-seed`) so DLT can filter simulator vs app REST traffic (`rest_app`); **replace_demo_rows** deletes only seeded demo rows for the targeted `user_id` values.

In [ ]:
dbutils.widgets.text("catalog", "users", "Catalog")
dbutils.widgets.text("schema", "ankur_nayyar", "Schema")
dbutils.widgets.dropdown("payload_size", "demo", ["standard", "demo"], "Payload size")
dbutils.widgets.dropdown(
    "seed_mode",
    "append",
    ["append", "replace_demo_rows"],
    "append = new rows only; replace_demo_rows = delete prior demo-notebook-seed rows for this user then insert",
)
dbutils.widgets.text("user_id", "", "user_id (empty = current_user())")
dbutils.widgets.dropdown(
    "multi_user_mode",
    "primary_plus_demo",
    ["single", "demo_cohort_only", "primary_plus_demo"],
    "single=primary only; demo_cohort_only=6 demo personas; primary_plus_demo=primary+6 personas",
)
dbutils.widgets.text("history_days", "60", "Days of synthetic history (1–730; activity + sleep + samples)")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
BRONZE = f"{CATALOG}.{SCHEMA}.wearables_zerobus"

uid = dbutils.widgets.get("user_id").strip()
USER_ID = uid if uid else spark.sql("SELECT current_user()").collect()[0][0]

print("Bronze table:", BRONZE)
print("user_id (primary):", USER_ID)
print("multi_user_mode:", dbutils.widgets.get("multi_user_mode"))
print("payload_size:", dbutils.widgets.get("payload_size"))
print("seed_mode:", dbutils.widgets.get("seed_mode"))

In [ ]:
import sys
from pathlib import Path

_nb_path = Path(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().lstrip("/")
)
_src_dir = _nb_path.parent.parent
for _p in (_src_dir / "endpoint-validation", _src_dir / "notebooks"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from bronze_seed_helpers import seed_target_user_ids, sql_in_list

SEED_USERS = seed_target_user_ids(USER_ID, dbutils.widgets.get("multi_user_mode"))
USER_IN_SQL = sql_in_list(SEED_USERS)

mode = dbutils.widgets.get("seed_mode")
if mode == "replace_demo_rows":
    spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")
    n = spark.sql(
        f"""
        SELECT COUNT(*) AS c FROM {BRONZE}
        WHERE user_id IN ({USER_IN_SQL})
          AND get_json_object(to_json(headers), '$.x-device-id') = 'demo-notebook-seed'
        """
    ).collect()[0]["c"]
    print(f"Deleting {n} prior demo rows for user_id IN ({len(SEED_USERS)} identities) …")
    spark.sql(
        f"""
        DELETE FROM {BRONZE}
        WHERE user_id IN ({USER_IN_SQL})
          AND get_json_object(to_json(headers), '$.x-device-id') = 'demo-notebook-seed'
        """
    )
    print("Done.")
else:
    print("seed_mode=append — skipping DELETE.")

In [ ]:
import sys
from pathlib import Path

# Repo / bundle layout: this notebook lives under .../src/notebooks/
nb_path = Path(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().lstrip("/")
)
src_dir = nb_path.parent.parent  # .../src
ep_dir = src_dir / "endpoint-validation"
sys.path.insert(0, str(ep_dir))
sys.path.insert(0, str(src_dir / "notebooks"))

from bronze_seed_helpers import build_seed_rows, seed_target_user_ids

_hd = dbutils.widgets.get("history_days").strip()
try:
    HISTORY_DAYS = int(_hd) if _hd else 60
except ValueError:
    HISTORY_DAYS = 60

seed_users = seed_target_user_ids(USER_ID, dbutils.widgets.get("multi_user_mode"))
rows = []
for su in seed_users:
    part = build_seed_rows(su, dbutils.widgets.get("payload_size"), history_days=HISTORY_DAYS)
    rows.extend(part)
print(f"Seeding {len(seed_users)} user(s) → {len(rows)} bronze rows.")

df = spark.createDataFrame(rows)
df.createOrReplaceTempView("_wearables_bronze_seed_stage")

spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")
spark.sql(
    f"""
    INSERT INTO {BRONZE} (
      record_id,
      ingested_at,
      body,
      headers,
      record_type,
      source_platform,
      user_id
    )
    SELECT
      record_id,
      ingested_at,
      parse_json(body_json) AS body,
      parse_json(headers_json) AS headers,
      record_type,
      source_platform,
      user_id
    FROM _wearables_bronze_seed_stage
    """
)

inserted = spark.sql(f"SELECT COUNT(*) AS c FROM {BRONZE} WHERE record_id IN (SELECT record_id FROM _wearables_bronze_seed_stage)").collect()[0]["c"]
print(f"Verified {inserted} rows visible in bronze (match stage record_ids).")

In [ ]:
from bronze_seed_helpers import sql_in_list

_uid_sql = sql_in_list(seed_users)
display(
    spark.sql(
        f"""
        SELECT user_id, record_type, COUNT(*) AS rows
        FROM {BRONZE}
        WHERE user_id IN ({_uid_sql})
          AND get_json_object(to_json(headers), '$.x-device-id') = 'demo-notebook-seed'
        GROUP BY user_id, record_type
        ORDER BY user_id, record_type
        """
    )
)

## Next steps (Databricks value path)

1. **Run your DLT / Lakeflow pipeline** that reads `wearables_bronze_table` so silver + gold refresh.
2. **Lakeview dashboard:** SQL Editor → open queries under `dashboards/wearables_apple_health/*.sql` → *Save* → *Create dashboard* → add **counter**, **trend**, **bar**, **table** visualizations.
3. **Operational polish:** enable **query tags** in jobs, use **Unity Catalog** lineage on pipeline tables, and (if allowed in your workspace) **Predictive Optimization** on the bronze schema — see `DASHBOARD_GUIDE.md` for a concise checklist.